# Telugu PowerExtra inference (3-seed joint mBERT)

Runs the 90-item `Telugu_PowerExtra` pool through the 3 real seed checkpoints (42/123/7).
CPU runtime is fine. When prompted, upload `annotation_pool.jsonl` from
`IdiomBERT/experiments/Evaluation/annotation_tool/data/Telugu_PowerExtra/`.

Downloads 3 files at the end: `test_predictions_new90_s{42,123,7}.jsonl`.

In [ ]:
!pip install -q transformers torch

In [ ]:
import json
from pathlib import Path

import torch
from google.colab import drive, files
from transformers import AutoTokenizer, AutoModel

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Idiomator_Research')
if not DRIVE_ROOT.exists():
    candidates = list(Path('/content/drive').rglob('Idiomator_Research'))
    if not candidates:
        raise SystemExit("Couldn't find 'Idiomator_Research' under /content/drive — check the mount.")
    DRIVE_ROOT = candidates[0]
print('Using DRIVE_ROOT:', DRIVE_ROOT)

# seed -> real checkpoint folder (seed 42 lives under its original experiment name, never renamed on Drive)
SEED_DIRS = {
    42:  DRIVE_ROOT / 'models' / 'en_es_hi_te' / 'joint_mbert',
    123: DRIVE_ROOT / 'models' / 'main_s123' / 'joint_mbert',
    7:   DRIVE_ROOT / 'models' / 'main_s7' / 'joint_mbert',
}
for seed, d in SEED_DIRS.items():
    assert (d / 'best_model' / 'model.safetensors').exists(), f'missing encoder for seed {seed}: {d}'
print('All 3 seed encoders found.')

In [ ]:
print('Upload Telugu_PowerExtra/annotation_pool.jsonl now:')
uploaded = files.upload()
pool_path = next(iter(uploaded.keys()))

examples = []
with open(pool_path, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            examples.append(json.loads(line))
print(f'Loaded {len(examples)} examples')

In [ ]:
LABEL2ID = {'literal': 0, 'idiomatic': 1}
ID2LABEL = {0: 'literal', 1: 'idiomatic'}
MAX_LEN = 128
BATCH_SIZE = 32
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# ---- same model/dataset classes as Evaluation/infer_shared_test.py ----

class JointIdiomModel(torch.nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.cls_head = torch.nn.Linear(hidden_size, 2)
        self.start_head = torch.nn.Linear(hidden_size, 1)
        self.end_head = torch.nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        seq_output = outputs.last_hidden_state
        cls_logits = self.cls_head(seq_output[:, 0, :])
        start_logits = self.start_head(seq_output).squeeze(-1)
        end_logits = self.end_head(seq_output).squeeze(-1)
        mask = attention_mask.bool()
        start_logits = start_logits.masked_fill(~mask, float('-inf'))
        end_logits = end_logits.masked_fill(~mask, float('-inf'))
        return cls_logits, start_logits, end_logits


def load_joint_model(joint_dir, device):
    best = Path(joint_dir) / 'best_model'
    model = JointIdiomModel(str(best))  # real encoder present, no base-model fallback needed
    heads = torch.load(best / 'task_heads.pt', map_location='cpu', weights_only=True)
    model.cls_head.load_state_dict(heads['cls_head'])
    model.start_head.load_state_dict(heads['start_head'])
    model.end_head.load_state_dict(heads['end_head'])
    return model.to(device)


def char_to_token_span(encoding, char_start, char_end, sentence):
    token_start = None
    for i in range(len(sentence)):
        t = encoding.char_to_token(i)
        if t is not None and i >= char_start:
            token_start = t
            break
    token_end = None
    for i in range(char_end - 1, -1, -1):
        t = encoding.char_to_token(i)
        if t is not None:
            token_end = t
            break
    if token_start is None or token_end is None:
        return None, None
    if token_start > token_end:
        token_end = token_start
    return token_start, token_end


def token_to_char_span(tokenizer, sentence, token_start, token_end, max_len):
    enc = tokenizer(sentence, max_length=max_len, truncation=True, return_offsets_mapping=True)
    offsets = enc['offset_mapping']
    if token_start >= len(offsets) or token_end >= len(offsets):
        return None, None
    return offsets[token_start][0], offsets[token_end][1]


def compute_overlap_f1(pred_start, pred_end, gold_start, gold_end):
    pred_set = set(range(int(pred_start), int(pred_end) + 1))
    gold_set = set(range(int(gold_start), int(gold_end) + 1))
    if not pred_set or not gold_set:
        return 0.0
    overlap = len(pred_set & gold_set)
    if overlap == 0:
        return 0.0
    p = overlap / len(pred_set)
    r = overlap / len(gold_set)
    return 2 * p * r / (p + r)


class SpanDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer, max_len):
        self.valid_examples = []
        self.input_ids = []
        self.attention_masks = []
        self.token_type_ids = []
        skipped = 0
        for ex in examples:
            sentence = ex['sentence']
            char_start, char_end = ex['span_start'], ex['span_end']
            encoding = tokenizer(sentence, max_length=max_len, padding='max_length', truncation=True, return_tensors='pt')
            enc_map = tokenizer(sentence, max_length=max_len, truncation=True, return_offsets_mapping=True)
            token_start, token_end = char_to_token_span(enc_map, char_start, char_end, sentence)
            if token_start is None or token_end is None:
                skipped += 1
                continue
            self.valid_examples.append(ex)
            self.input_ids.append(encoding['input_ids'].squeeze(0))
            self.attention_masks.append(encoding['attention_mask'].squeeze(0))
            tid = encoding.get('token_type_ids')
            self.token_type_ids.append(tid.squeeze(0) if tid is not None else torch.zeros(encoding['input_ids'].shape[1], dtype=torch.long))
        if skipped:
            print(f'  Skipped {skipped} examples with failed span alignment')

    def __len__(self):
        return len(self.valid_examples)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'token_type_ids': self.token_type_ids[idx],
        }

In [ ]:
import numpy as np


def run_joint(examples, joint_dir, device, out_path):
    best = Path(joint_dir) / 'best_model'
    tokenizer = AutoTokenizer.from_pretrained(str(best))
    model = load_joint_model(joint_dir, device)
    model.eval()

    dataset = SpanDataset(examples, tokenizer, MAX_LEN)
    loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE)
    valid = dataset.valid_examples
    labels = [LABEL2ID[ex['idiomaticity']] for ex in valid]

    records = []
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            cls_logits, start_logits, end_logits = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
                token_type_ids=batch['token_type_ids'].to(device),
            )
            cls_preds = torch.argmax(cls_logits, dim=-1).cpu().numpy()
            pred_starts = torch.argmax(start_logits, dim=-1).cpu().numpy()
            pred_ends = torch.argmax(end_logits, dim=-1).cpu().numpy()

            batch_start = batch_idx * BATCH_SIZE
            for i in range(len(cls_preds)):
                ex_idx = batch_start + i
                if ex_idx >= len(valid):
                    break
                ex = valid[ex_idx]
                pred_s, pred_e = int(pred_starts[i]), int(pred_ends[i])
                if pred_e < pred_s:
                    pred_e = pred_s
                pred_char_s, pred_char_e = token_to_char_span(tokenizer, ex['sentence'], pred_s, pred_e, MAX_LEN)
                if pred_char_s is None:
                    pred_char_s, pred_char_e = 0, 0
                records.append({
                    **ex,
                    'pred_idiomaticity': ID2LABEL[int(cls_preds[i])],
                    'correct': bool(cls_preds[i] == labels[ex_idx]),
                    'pred_span_start': pred_char_s,
                    'pred_span_end': pred_char_e,
                    'exact_match': bool(pred_char_s == ex['span_start'] and pred_char_e == ex['span_end']),
                    'overlap_f1': round(compute_overlap_f1(pred_char_s, pred_char_e, ex['span_start'], ex['span_end']), 4),
                })

    print(f"  seed dir {joint_dir}: cls_acc={np.mean([r['correct'] for r in records]):.4f} "
          f"exact={np.mean([r['exact_match'] for r in records]):.4f} "
          f"overlap={np.mean([r['overlap_f1'] for r in records]):.4f}")

    with open(out_path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'  Saved -> {out_path}')


for seed, joint_dir in SEED_DIRS.items():
    print(f'\n=== Seed {seed} ===')
    out_path = f'test_predictions_new90_s{seed}.jsonl'
    run_joint(examples, joint_dir, device, out_path)
    files.download(out_path)

print('\nDone. Send back the 3 downloaded test_predictions_new90_s{42,123,7}.jsonl files.')